In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
# Fix sentence-transformers compatibility issue
import subprocess
import sys
import os

print("🔧 Fixing sentence-transformers compatibility...")

# The issue is with huggingface_hub version compatibility
# We need to install compatible versions

# First, uninstall problematic packages
print("Removing incompatible packages...")
packages_to_remove = [
    'sentence-transformers',
    'huggingface_hub',
    'transformers'
]

for package in packages_to_remove:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', package], 
                            capture_output=True)
        print(f"✅ Removed {package}")
    except:
        print(f"⚠️ {package} not found or already removed")

# Install compatible versions in correct order
print("Installing compatible versions...")
compatible_packages = [
    'huggingface_hub==0.16.4',      # Compatible version
    'transformers==4.21.3',         # Compatible with sentence-transformers
    'sentence-transformers==2.2.2'  # Latest that works with the above
]

for package in compatible_packages:
    print(f"Installing {package}...")
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '--no-deps'], 
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        print(f"✅ Successfully installed {package}")
    except Exception as e:
        print(f"❌ Error installing {package}: {e}")
        # Try without --no-deps
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', package], 
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            print(f"✅ Installed {package} (with dependencies)")
        except Exception as e2:
            print(f"❌ Final error: {e2}")

# Install any missing dependencies
missing_deps = ['tokenizers', 'tqdm', 'torch', 'torchvision', 'numpy', 'scikit-learn', 'scipy', 'nltk', 'pillow']
for dep in missing_deps:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', dep], 
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    except:
        pass  # Probably already installed

print("\n🔄 IMPORTANT: Restart your kernel/runtime now!")
print("In Kaggle: Runtime → Restart Session")
print("Then run the next cell to verify the installation.")

🔧 Fixing sentence-transformers compatibility...
Removing incompatible packages...
⚠️ sentence-transformers not found or already removed
⚠️ huggingface_hub not found or already removed
⚠️ transformers not found or already removed
Installing compatible versions...
Installing huggingface_hub==0.16.4...
✅ Successfully installed huggingface_hub==0.16.4
Installing transformers==4.21.3...
✅ Successfully installed transformers==4.21.3
Installing sentence-transformers==2.2.2...
✅ Successfully installed sentence-transformers==2.2.2

🔄 IMPORTANT: Restart your kernel/runtime now!
In Kaggle: Runtime → Restart Session
Then run the next cell to verify the installation.


In [6]:
# Import all required libraries with fallback options
print("📦 Importing libraries with fallback handling...")

import pandas as pd
import numpy as np
import os
import re
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer

# Deep learning libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Try to import sentence transformers with fallback
sentence_transformers_available = False
try:
    from sentence_transformers import SentenceTransformer
    sentence_transformers_available = True
    print("✅ SentenceTransformer imported successfully!")
except ImportError as e:
    print(f"⚠️ SentenceTransformer import failed: {e}")
    print("🔄 Will use TF-IDF vectorization as backup")
    SentenceTransformer = None


# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
else:
    print("💻 Using CPU")

if sentence_transformers_available:
    print("✅ All imports successful with SentenceTransformer!")
else:
    print("⚠️ Running with TF-IDF backup (still very effective!)")

print("🎯 Ready to proceed!")

📦 Importing libraries with fallback handling...
⚠️ SentenceTransformer import failed: cannot import name 'cached_download' from 'huggingface_hub' (/usr/local/lib/python3.11/dist-packages/huggingface_hub/__init__.py)
🔄 Will use TF-IDF vectorization as backup
🚀 GPU: Tesla T4
⚠️ Running with TF-IDF backup (still very effective!)
🎯 Ready to proceed!


In [7]:
# Load and preprocess data - Kaggle version
# Kaggle input paths (adjust these based on your dataset name)
# Replace 'your-dataset-name' with your actual Kaggle dataset name
INPUT_PATH = '/kaggle/input/amazon3'

# Try to auto-detect the dataset folder
dataset_folders = [d for d in os.listdir(INPUT_PATH) if os.path.isdir(os.path.join(INPUT_PATH, d))]
if dataset_folders:
    DATA_PATH = os.path.join(INPUT_PATH, dataset_folders[0])
    print(f"📁 Using dataset folder: {dataset_folders[0]}")
else:
    # Fallback - you may need to adjust this path
    DATA_PATH = INPUT_PATH
    print("📁 Using root input folder")

# File paths for Kaggle
TRAIN_CSV = os.path.join(DATA_PATH, 'train.csv')
TEST_CSV = os.path.join(DATA_PATH, 'sample_test.csv')
TEST_OUT_CSV = os.path.join(DATA_PATH, 'sample_test_out.csv')
FINAL_TEST_CSV = os.path.join(DATA_PATH, 'test.csv')

# Check which files exist
print("📂 Checking for data files:")
for file_path, name in [(TRAIN_CSV, 'train.csv'), (TEST_CSV, 'sample_test.csv'), 
                        (TEST_OUT_CSV, 'sample_test_out.csv'), (FINAL_TEST_CSV, 'test.csv')]:
    if os.path.exists(file_path):
        print(f"✅ Found: {name}")
    else:
        print(f"❌ Missing: {name}")

# Load datasets
print("\n📂 Loading datasets...")
df_train_full = pd.read_csv(TRAIN_CSV)
df_test = pd.read_csv(TEST_CSV) if os.path.exists(TEST_CSV) else None
df_test_out = pd.read_csv(TEST_OUT_CSV) if os.path.exists(TEST_OUT_CSV) else None

print(f"✅ Training records: {len(df_train_full):,}")
if df_test is not None:
    print(f"✅ Sample test records: {len(df_test):,}")
if df_test_out is not None:
    print(f"✅ Sample test output records: {len(df_test_out):,}")

# Display sample data
print("\n📋 Sample training data:")
print(df_train_full.head(2))

📁 Using root input folder
📂 Checking for data files:
✅ Found: train.csv
❌ Missing: sample_test.csv
❌ Missing: sample_test_out.csv
✅ Found: test.csv

📂 Loading datasets...
✅ Training records: 75,000

📋 Sample training data:
   sample_id                                    catalog_content  \
0      33127  Item Name: La Victoria Green Taco Sauce Mild, ...   
1     198967  Item Name: Salerno Cookies, The Original Butte...   

                                          image_link  price  
0  https://m.media-amazon.com/images/I/51mo8htwTH...   4.89  
1  https://m.media-amazon.com/images/I/71YtriIHAA...  13.12  


In [9]:
# Advanced text preprocessing and feature engineering
def advanced_text_preprocessing(text):
    """Enhanced text preprocessing for better feature extraction"""
    if pd.isna(text) or text is None:
        return ""
    
    text = str(text).lower().strip()
    
    # Remove price mentions to avoid data leakage
    text = re.sub(r'\$[\d,]+\.?\d*', '', text)
    text = re.sub(r'price[:\s]*[\d,]+\.?\d*', '', text)
    text = re.sub(r'\b\d+\.\d+\b', '', text)  # Remove decimal numbers
    
    # Clean and normalize
    text = re.sub(r'\b(\w+)\s+brand\b', r'\1', text)  # Normalize brand mentions
    text = re.sub(r'[^\w\s]', ' ', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text)  # Normalize whitespace
    
    return text.strip()

def extract_text_features(text):
    """Extract statistical features from text"""
    if pd.isna(text) or text == "":
        return {
            'char_count': 0, 'word_count': 0, 'sentence_count': 0,
            'avg_word_len': 0, 'capital_count': 0, 'digit_count': 0,
            'special_char_count': 0
        }
    
    words = text.split()
    return {
        'char_count': len(text),
        'word_count': len(words),
        'sentence_count': text.count('.') + text.count('!') + text.count('?'),
        'avg_word_len': np.mean([len(word) for word in words]) if words else 0,
        'capital_count': sum(1 for c in str(text) if c.isupper()),
        'digit_count': sum(1 for c in str(text) if c.isdigit()),
        'special_char_count': len(re.findall(r'[^\w\s]', str(text)))
    }

# Apply preprocessing to training data
print("🧹 Preprocessing text data...")
df_train_full['processed_content'] = df_train_full['catalog_content'].apply(advanced_text_preprocessing)

# Only process test data if it exists
if df_test is not None:
    df_test['processed_content'] = df_test['catalog_content'].apply(advanced_text_preprocessing)
    print("✅ Sample test data preprocessed")
else:
    print("⚠️ Sample test data not available - skipping preprocessing")

# Extract statistical features for training data
print("📊 Extracting text features...")
train_text_features = pd.DataFrame([extract_text_features(text) for text in df_train_full['processed_content']])

# Add features to training dataframe
for col in train_text_features.columns:
    df_train_full[col] = train_text_features[col]

# Only extract features for test data if it exists
if df_test is not None:
    test_text_features = pd.DataFrame([extract_text_features(text) for text in df_test['processed_content']])
    # Add features to test dataframe
    for col in test_text_features.columns:
        df_test[col] = test_text_features[col]
    print("✅ Sample test features extracted")
else:
    print("⚠️ Sample test features skipped (no test data)")

print("✅ Text preprocessing complete!")
print(f"📈 Feature columns: {list(train_text_features.columns)}")

# Show data availability status
print(f"\n📊 Data Status:")
print(f"Training data: ✅ Available ({len(df_train_full):,} records)")
print(f"Sample test data: {'✅ Available' if df_test is not None else '❌ Not available'}")
print(f"Sample test labels: {'✅ Available' if df_test_out is not None else '❌ Not available'}")

🧹 Preprocessing text data...
⚠️ Sample test data not available - skipping preprocessing
📊 Extracting text features...
⚠️ Sample test features skipped (no test data)
✅ Text preprocessing complete!
📈 Feature columns: ['char_count', 'word_count', 'sentence_count', 'avg_word_len', 'capital_count', 'digit_count', 'special_char_count']

📊 Data Status:
Training data: ✅ Available (75,000 records)
Sample test data: ❌ Not available
Sample test labels: ❌ Not available


In [10]:
# Target transformation and data preparation - Fixed version
def transform_target(y):
    """Optimized target transformation for price data"""
    return np.sqrt(np.log1p(y))

def inverse_transform_target(y_transformed):
    """Inverse transform predictions back to original scale"""
    return np.expm1(np.square(y_transformed))

# Apply target transformation
print("🎯 Transforming target variable...")
df_train_full['transformed_price'] = transform_target(df_train_full['price'])

# Create stratified train-validation split
df_train_full['price_bin'] = pd.qcut(df_train_full['price'], q=5, labels=False, duplicates='drop')
df_train, df_val = train_test_split(
    df_train_full, 
    test_size=0.15, 
    random_state=42, 
    stratify=df_train_full['price_bin']
)

print(f"📚 Training samples: {len(df_train):,}")
print(f"📝 Validation samples: {len(df_val):,}")

# Load model or use TF-IDF backup
print("🤖 Setting up text vectorization...")

if sentence_transformers_available and SentenceTransformer is not None:
    print("Attempting to load SentenceTransformer...")
    try:
        # Try different models in order of preference
        models_to_try = [
            'sentence-transformers/all-MiniLM-L6-v2',  # Smaller, more compatible
            'all-MiniLM-L6-v2',  # Alternative naming
            'paraphrase-MiniLM-L3-v2'  # Even smaller backup
        ]
        
        sentence_model = None
        for model_name in models_to_try:
            try:
                print(f"   Trying {model_name}...")
                sentence_model = SentenceTransformer(model_name)
                print(f"✅ Successfully loaded {model_name}!")
                break
            except Exception as e:
                print(f"   Failed: {e}")
                continue
        
        if sentence_model is None:
            raise Exception("All SentenceTransformer models failed")
            
    except Exception as e:
        print(f"❌ All SentenceTransformer attempts failed: {e}")
        sentence_model = None
        
else:
    print("SentenceTransformer not available, using TF-IDF")
    sentence_model = None

# Generate embeddings or use TF-IDF
print("🔢 Generating text representations...")

if sentence_model is not None:
    print("   Using SentenceTransformer embeddings...")
    
    # Process in smaller batches to avoid memory issues
    def encode_in_batches(texts, batch_size=8):
        embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            batch_embeddings = sentence_model.encode(batch, show_progress_bar=False)
            embeddings.extend(batch_embeddings)
        return np.array(embeddings)
    
    train_embeddings = encode_in_batches(df_train['processed_content'].tolist())
    val_embeddings = encode_in_batches(df_val['processed_content'].tolist())
    
    if df_test is not None:
        test_embeddings = encode_in_batches(df_test['processed_content'].tolist())
    else:
        test_embeddings = None
        
    print(f"📐 Embedding dimension: {train_embeddings.shape[1]}")
    
else:
    # Robust TF-IDF backup
    print("   Using TF-IDF vectorization (highly effective backup)...")
    
    # Use comprehensive TF-IDF settings
    tfidf = TfidfVectorizer(
        max_features=2000,  # Increased features
        stop_words='english',
        ngram_range=(1,3),  # Include trigrams
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )
    
    # Fit on training data and transform all sets
    train_embeddings = tfidf.fit_transform(df_train['processed_content']).toarray()
    val_embeddings = tfidf.transform(df_val['processed_content']).toarray()
    
    if df_test is not None:
        test_embeddings = tfidf.transform(df_test['processed_content']).toarray()
    else:
        test_embeddings = None
        
    print(f"📐 TF-IDF dimension: {train_embeddings.shape[1]}")

# Prepare statistical features
feature_cols = ['char_count', 'word_count', 'sentence_count', 'avg_word_len', 
                'capital_count', 'digit_count', 'special_char_count']

# Normalize features
scaler = StandardScaler()
train_stat_features = scaler.fit_transform(df_train[feature_cols])
val_stat_features = scaler.transform(df_val[feature_cols])
if df_test is not None:
    test_stat_features = scaler.transform(df_test[feature_cols])
else:
    test_stat_features = None

print("✅ Text representation and feature preparation complete!")
print(f"🎯 Ready for model training with {train_embeddings.shape[1]} text features + {len(feature_cols)} statistical features")

🎯 Transforming target variable...
📚 Training samples: 63,750
📝 Validation samples: 11,250
🤖 Setting up text vectorization...
SentenceTransformer not available, using TF-IDF
🔢 Generating text representations...
   Using TF-IDF vectorization (highly effective backup)...
📐 TF-IDF dimension: 2000
✅ Text representation and feature preparation complete!
🎯 Ready for model training with 2000 text features + 7 statistical features


In [11]:
# Advanced Neural Network Model - Kaggle optimized
class AdvancedPricePredictor(nn.Module):
    def __init__(self, embedding_dim, stat_features_dim):
        super(AdvancedPricePredictor, self).__init__()
        
        # Embedding processing (smaller for Kaggle memory)
        self.embedding_layers = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Statistical features processing
        self.stat_layers = nn.Sequential(
            nn.Linear(stat_features_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Combined processing
        self.combined_layers = nn.Sequential(
            nn.Linear(128 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.BatchNorm1d(64),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1)
        )
        
    def forward(self, embeddings, stat_features):
        emb_out = self.embedding_layers(embeddings)
        stat_out = self.stat_layers(stat_features)
        combined = torch.cat([emb_out, stat_out], dim=1)
        output = self.combined_layers(combined)
        return output

# Custom Dataset
class PriceDataset(Dataset):
    def __init__(self, embeddings, stat_features, labels=None):
        self.embeddings = torch.FloatTensor(embeddings)
        self.stat_features = torch.FloatTensor(stat_features)
        self.labels = torch.FloatTensor(labels) if labels is not None else None

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        item = {
            'embeddings': self.embeddings[idx],
            'stat_features': self.stat_features[idx]
        }
        if self.labels is not None:
            item['labels'] = self.labels[idx]
        return item

# Create datasets
train_dataset = PriceDataset(
    train_embeddings, 
    train_stat_features,
    df_train['transformed_price'].values
)

val_dataset = PriceDataset(
    val_embeddings, 
    val_stat_features,
    df_val['transformed_price'].values
)

if test_embeddings is not None:
    test_dataset = PriceDataset(test_embeddings, test_stat_features)
else:
    test_dataset = None

# Create data loaders (Kaggle optimized)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
if test_dataset is not None:
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)
else:
    test_loader = None

# Initialize model
model = AdvancedPricePredictor(
    embedding_dim=train_embeddings.shape[1],
    stat_features_dim=len(feature_cols)
).to(device)

# Optimizer and scheduler (lighter for Kaggle)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
criterion = nn.MSELoss()

print("🧠 Model initialized successfully!")
print(f"📊 Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# SMAPE calculation
def smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error"""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator != 0
    
    if np.sum(mask) == 0:
        return 0.0
    
    numerator = np.abs(y_pred - y_true)[mask]
    denominator = denominator[mask]
    
    return np.mean(numerator / denominator) * 100

print("🎯 Ready for training!")

🧠 Model initialized successfully!
📊 Model parameters: 558,465
🎯 Ready for training!


In [12]:
# Training loop - Kaggle optimized
print("🚀 Starting training...")

epochs = 15  # Reduced for Kaggle time limits
best_smape = float('inf')
train_losses = []
val_smapes = []

for epoch in range(epochs):
    # Training
    model.train()
    epoch_loss = 0
    num_batches = 0
    
    for batch in train_loader:
        embeddings = batch['embeddings'].to(device)
        stat_features = batch['stat_features'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        outputs = model(embeddings, stat_features)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_train_loss = epoch_loss / num_batches
    train_losses.append(avg_train_loss)
    
    # Validation
    model.eval()
    val_predictions = []
    val_actuals = []
    
    with torch.no_grad():
        for batch in val_loader:
            embeddings = batch['embeddings'].to(device)
            stat_features = batch['stat_features'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(embeddings, stat_features)
            val_predictions.extend(outputs.squeeze().cpu().numpy())
            val_actuals.extend(labels.cpu().numpy())
    
    # Convert back to original scale and calculate SMAPE
    val_preds_original = inverse_transform_target(np.array(val_predictions))
    val_actuals_original = inverse_transform_target(np.array(val_actuals))
    
    current_smape = smape(val_actuals_original, val_preds_original)
    val_smapes.append(current_smape)
    
    # Update learning rate
    scheduler.step()
    
    # Save best model
    if current_smape < best_smape:
        best_smape = current_smape
        torch.save(model.state_dict(), 'best_model_kaggle.pth')
    
    # Print progress
    if epoch % 3 == 0 or epoch == epochs - 1:
        print(f'Epoch {epoch+1:2d}/{epochs}: Train Loss: {avg_train_loss:.6f}, Val SMAPE: {current_smape:.4f}%, Best: {best_smape:.4f}%')

print(f"\n🎯 Training complete! Best SMAPE: {best_smape:.4f}%")

🚀 Starting training...
Epoch  1/15: Train Loss: 0.385797, Val SMAPE: 64.9990%, Best: 64.9990%
Epoch  4/15: Train Loss: 0.079333, Val SMAPE: 60.4697%, Best: 60.4697%
Epoch  7/15: Train Loss: 0.065150, Val SMAPE: 57.5509%, Best: 57.5509%
Epoch 10/15: Train Loss: 0.055634, Val SMAPE: 56.8749%, Best: 56.8749%
Epoch 13/15: Train Loss: 0.048606, Val SMAPE: 56.8414%, Best: 56.8414%
Epoch 15/15: Train Loss: 0.045759, Val SMAPE: 57.7645%, Best: 56.8414%

🎯 Training complete! Best SMAPE: 56.8414%


In [13]:
# Final predictions and submission - Kaggle version
print("🔮 Making final predictions...")

# Load best model
model.load_state_dict(torch.load('best_model_kaggle.pth'))
model.eval()

# Evaluate on sample test set if available
if test_loader is not None and df_test_out is not None:
    print("📊 Evaluating on sample test set...")
    
    # Neural network predictions on sample test set
    nn_predictions = []
    with torch.no_grad():
        for batch in test_loader:
            embeddings = batch['embeddings'].to(device)
            stat_features = batch['stat_features'].to(device)
            outputs = model(embeddings, stat_features)
            nn_predictions.extend(outputs.squeeze().cpu().numpy())

    # Convert to original scale
    nn_prices = inverse_transform_target(np.array(nn_predictions))
    nn_prices = np.maximum(nn_prices, 0.01)

    # Random Forest ensemble
    print("🌲 Training Random Forest ensemble...")
    rf_features_train = np.hstack([train_embeddings, df_train[feature_cols].values])
    rf_features_test = np.hstack([test_embeddings, df_test[feature_cols].values])

    rf_model = RandomForestRegressor(
        n_estimators=100,  # Reduced for Kaggle
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(rf_features_train, df_train['price'].values)
    rf_prices = rf_model.predict(rf_features_test)
    rf_prices = np.maximum(rf_prices, 0.01)

    # Ensemble predictions
    ensemble_prices = 0.7 * nn_prices + 0.3 * rf_prices

    # Evaluate
    df_test['nn_predicted_price'] = nn_prices
    df_test['rf_predicted_price'] = rf_prices  
    df_test['ensemble_predicted_price'] = ensemble_prices

    results_df = pd.merge(
        df_test[['sample_id', 'nn_predicted_price', 'rf_predicted_price', 'ensemble_predicted_price']], 
        df_test_out, 
        on='sample_id'
    )

    nn_smape = smape(results_df['price'], results_df['nn_predicted_price'])
    rf_smape = smape(results_df['price'], results_df['rf_predicted_price'])
    ensemble_smape = smape(results_df['price'], results_df['ensemble_predicted_price'])

    print(f"🧠 Neural Network SMAPE: {nn_smape:.4f}%")
    print(f"🌲 Random Forest SMAPE: {rf_smape:.4f}%")
    print(f"🎯 Ensemble SMAPE: {ensemble_smape:.4f}%")

    # Choose best model
    best_smape_test = min(nn_smape, rf_smape, ensemble_smape)
    if ensemble_smape == best_smape_test:
        best_model_name = "Ensemble"
    elif nn_smape == best_smape_test:
        best_model_name = "Neural Network"
    else:
        best_model_name = "Random Forest"
        
    print(f"🏆 Best model: {best_model_name} (SMAPE: {best_smape_test:.4f}%)")

# Create final submission for actual test set
if os.path.exists(FINAL_TEST_CSV):
    df_final_test = pd.read_csv(FINAL_TEST_CSV)
    print(f"\n📁 Final test set: {len(df_final_test):,} records")

    # Preprocess final test data
    df_final_test['processed_content'] = df_final_test['catalog_content'].apply(advanced_text_preprocessing)

    # Extract features
    final_text_features = pd.DataFrame([extract_text_features(text) for text in df_final_test['processed_content']])
    for col in final_text_features.columns:
        df_final_test[col] = final_text_features[col]

    # Generate embeddings
    print("🔢 Generating final test embeddings...")
    if sentence_model is not None:
        final_embeddings = sentence_model.encode(
            df_final_test['processed_content'].tolist(), 
            show_progress_bar=True,
            batch_size=16
        )
    else:
        # Use TF-IDF if sentence transformers failed
        final_embeddings = tfidf.transform(df_final_test['processed_content']).toarray()

    # Normalize features
    final_stat_features = scaler.transform(df_final_test[feature_cols])

    # Create dataset and loader
    final_dataset = PriceDataset(final_embeddings, final_stat_features)
    final_loader = DataLoader(final_dataset, batch_size=32, shuffle=False, num_workers=0)

    # Neural network predictions
    final_nn_preds = []
    with torch.no_grad():
        for batch in final_loader:
            embeddings = batch['embeddings'].to(device)
            stat_features = batch['stat_features'].to(device)
            outputs = model(embeddings, stat_features)
            final_nn_preds.extend(outputs.squeeze().cpu().numpy())

    final_nn_prices = inverse_transform_target(np.array(final_nn_preds))
    final_nn_prices = np.maximum(final_nn_prices, 0.01)

    # Random Forest predictions (if we have RF model from evaluation)
    if 'rf_model' in locals():
        final_rf_features = np.hstack([final_embeddings, df_final_test[feature_cols].values])
        final_rf_prices = rf_model.predict(final_rf_features)
        final_rf_prices = np.maximum(final_rf_prices, 0.01)
        
        # Final ensemble
        submission_prices = 0.7 * final_nn_prices + 0.3 * final_rf_prices
        model_used = "Ensemble"
    else:
        submission_prices = final_nn_prices
        model_used = "Neural Network"

    # Create Kaggle submission format
    submission_df = pd.DataFrame({
        'sample_id': df_final_test['sample_id'],  # Adjust column name if needed
        'price': submission_prices
    })

    # Ensure positive prices
    submission_df['price'] = submission_df['price'].apply(lambda x: max(0.01, float(x)))

    # Save submission in Kaggle working directory
    submission_df.to_csv('submission.csv', index=False)
    print(f"✅ Kaggle submission saved as 'submission.csv'")
    
    # Also save a backup with timestamp
    import datetime
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_filename = f'submission_backup_{timestamp}.csv'
    submission_df.to_csv(backup_filename, index=False)
    print(f"💾 Backup saved as '{backup_filename}'")

    print(f"\n📈 Final Statistics:")
    print(f"Model used: {model_used}")
    print(f"Mean price: ${submission_df['price'].mean():.2f}")
    print(f"Median price: ${submission_df['price'].median():.2f}")
    print(f"Min price: ${submission_df['price'].min():.2f}")
    print(f"Max price: ${submission_df['price'].max():.2f}")

    if 'best_smape_test' in locals():
        print(f"\n🎯 Performance Summary:")
        print(f"Sample Test SMAPE: {best_smape_test:.4f}%")
        print(f"Target: ~35%")

        if best_smape_test <= 35:
            print("🎉 EXCELLENT! Target achieved!")
        elif best_smape_test <= 40:
            print("🔥 Very close to target!")
        else:
            print("⚡ Good progress!")

    print("\n📋 Top 10 predictions:")
    print(submission_df.head(10))

else:
    print("❌ Final test set not found. Please ensure 'test.csv' is in your Kaggle input data.")

print("\n🎊 Kaggle submission ready!")
print("📤 Upload 'submission.csv' to your Kaggle competition!")

🔮 Making final predictions...

📁 Final test set: 75,000 records
🔢 Generating final test embeddings...
✅ Kaggle submission saved as 'submission.csv'
💾 Backup saved as 'submission_backup_20251013_153739.csv'

📈 Final Statistics:
Model used: Neural Network
Mean price: $17.29
Median price: $12.34
Min price: $0.64
Max price: $6586.63

📋 Top 10 predictions:
   sample_id      price
0     100179  15.244100
1     245611   9.608238
2     146263  26.290634
3      95658   9.126888
4      36806  26.653847
5     148239   8.324862
6      92659  14.239319
7       3780   5.659627
8     196940   5.102213
9      20472   5.894013

🎊 Kaggle submission ready!
📤 Upload 'submission.csv' to your Kaggle competition!
